

*"The curious task of economics is to demonstrate to men how little they really know about what they imagine they can design."* - Friedrich Hayek

*"I learned very early the difference between knowing the name of something and knowing something."* - Richard Feynman

---


Traditional economics assumes humans are perfectly rational, have unlimited computational power, and always act in their own best interest. But as anyone who has ever bought something they didn't need, procrastinated on important tasks, or held onto losing stocks too long knows, this assumption is... well, let's just say it's optimistic.

Behavioral economics bridges the gap between the elegant mathematical models of traditional economics and the messy, wonderful reality of human behavior. It's where psychology meets economics, where laboratory experiments inform policy, and where understanding cognitive biases can literally save lives and billions of dollars.

This notebook will take you on a comprehensive journey through the field, combining rigorous mathematical foundations with intuitive explanations, interactive demonstrations, and cutting-edge research. We'll explore not just what people do, but why they do it, and how we can design better systems that account for human psychology.


This isn't just another economics textbook. We'll:
- **Start with intuition** before diving into mathematics
- **Use interactive simulations** to see theories in action  
- **Examine real research** with actual data and findings
- **Build agent-based models** to understand emergent behaviors
- **Connect theory to policy** with real-world applications
- **Maintain mathematical rigor** while staying accessible


1. **Foundations**: Prospect Theory and the Value Function
2. **Loss Aversion**: Why Losses Loom Larger Than Gains
3. **Probability Weighting**: How We Distort Uncertainty
4. **Mental Accounting**: The Psychology of Money
5. **Anchoring and Adjustment**: The Power of First Impressions
6. **Framing Effects**: How Context Shapes Decisions
7. **Time Preferences**: Present Bias and Hyperbolic Discounting
8. **Social Preferences**: Fairness, Reciprocity, and Altruism
9. **Nudges and Choice Architecture**: Designing Better Decisions
10. **Market Anomalies**: When Behavioral Biases Meet Finance
11. **Neuroeconomics**: The Brain on Economics
12. **Policy Applications**: Behavioral Insights in Government

Let's begin this journey into the human mind and its beautiful, irrational complexity.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown, Image
import seaborn as sns
from scipy import optimize, stats
from scipy.special import expit
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 11

colors = {
    'gain': '#2E8B57',      # Sea green
    'loss': '#DC143C',      # Crimson  
    'neutral': '#4682B4',   # Steel blue
    'rational': '#708090',  # Slate gray
    'behavioral': '#FF6347', # Tomato
    'highlight': '#FFD700'  # Gold
}

import sys
sys.path.append('../utils')
sys.path.append('../simulations')

from economic_functions import *
from prospect_theory_agents import *
from market_simulation import *

print("🧠 Behavioral Economics Toolkit Loaded Successfully!")
print("📊 Ready to explore the fascinating world of human decision-making")




*"The concept of loss aversion is certainly the most significant contribution of psychology to behavioral economics."* - Daniel Kahneman


In 1979, Daniel Kahneman and Amos Tversky published a paper that would fundamentally change how we think about human decision-making. Their **Prospect Theory** challenged the dominant Expected Utility Theory and introduced concepts that explain countless "irrational" behaviors we observe every day.


**Expected Utility Theory** says people maximize:
$$EU = \sum_{i} p_i \cdot u(x_i)$$

Where $p_i$ is the probability of outcome $x_i$ and $u(x_i)$ is the utility of that outcome.

**Prospect Theory** says people actually maximize:
$$V = \sum_{i} w(p_i) \cdot v(x_i - r)$$

Where:
- $w(p_i)$ is the **decision weight** (not the objective probability)
- $v(x_i - r)$ is the **value function** relative to a reference point $r$
- The value function has different curvatures for gains and losses


The prospect theory value function has several key properties:

1. **Reference Dependence**: Outcomes are evaluated relative to a reference point, not absolute wealth
2. **Loss Aversion**: Losses loom larger than equivalent gains ($\lambda > 1$)
3. **Diminishing Sensitivity**: Both gains and losses show decreasing marginal impact

Mathematically:
$$v(x) = \begin{cases}
x^\alpha & \text{if } x \geq 0 \text{ (gains)} \\
-\lambda(-x)^\beta & \text{if } x < 0 \text{ (losses)}
\end{cases}$$

**Typical Parameter Values** (Tversky & Kahneman, 1992):
- $\alpha = 0.88$ (concavity for gains)
- $\beta = 0.88$ (convexity for losses)  
- $\lambda = 2.25$ (loss aversion coefficient)


People don't use objective probabilities in decisions. Instead, they apply a weighting function:

$$w(p) = \frac{p^\gamma}{(p^\gamma + (1-p)^\gamma)^{1/\gamma}}$$

With typical value $\gamma = 0.61$, this creates:
- **Overweighting** of small probabilities (lottery tickets, insurance)
- **Underweighting** of moderate to high probabilities
- **Certainty effect**: Disproportionate weight on certain outcomes


**Kahneman & Tversky (1984)** - The Asian Disease Problem:
- **Gain Frame**: "Save 200 lives" vs "1/3 chance save 600, 2/3 chance save 0"
- **Loss Frame**: "400 die" vs "1/3 chance 0 die, 2/3 chance 600 die"
- Same outcomes, different choices based on framing!

**Tversky & Kahneman (1991)** - Loss Aversion Measurement:
- Median loss aversion coefficient: $\lambda = 2.25$
- Range across studies: 1.5 to 4.0
- Robust across cultures and contexts


In [ ]:

def comprehensive_prospect_theory_demo():
    """
    Advanced interactive demonstration of prospect theory with research comparisons
    """
    
    tab_contents = []
    
    @widgets.interact(
        alpha=widgets.FloatSlider(value=0.88, min=0.1, max=1.0, step=0.01, 
                                 description='α (gain curvature)', style={'description_width': 'initial'}),
        beta=widgets.FloatSlider(value=0.88, min=0.1, max=1.0, step=0.01,
                                description='β (loss curvature)', style={'description_width': 'initial'}),
        lambda_param=widgets.FloatSlider(value=2.25, min=1.0, max=5.0, step=0.05,
                                        description='λ (loss aversion)', style={'description_width': 'initial'}),
        gamma=widgets.FloatSlider(value=0.61, min=0.1, max=1.0, step=0.01,
                                 description='γ (prob. weighting)', style={'description_width': 'initial'}),
        show_research=widgets.Checkbox(value=True, description='Show Research Benchmarks')
    )
    def plot_value_function(alpha, beta, lambda_param, gamma, show_research):
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        
        x = np.linspace(-100, 100, 1000)
        v = prospect_value_function(x, alpha, beta, lambda_param)
        
        ax1.plot(x, v, 'b-', linewidth=3, label='Current Parameters')
        
        if show_research:
            v_tk = prospect_value_function(x, 0.88, 0.88, 2.25)
            ax1.plot(x, v_tk, 'r--', linewidth=2, alpha=0.7, label='Tversky & Kahneman (1992)')
            
        ax1.axhline(y=0, color='k', linestyle='-', alpha=0.3)
        ax1.axvline(x=0, color='k', linestyle='-', alpha=0.3)
        ax1.set_xlabel('Outcome (relative to reference point)')
        ax1.set_ylabel('Subjective Value')
        ax1.set_title('Prospect Theory Value Function')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        p = np.linspace(0.01, 0.99, 100)
        w = probability_weighting_function(p, gamma)
        
        ax2.plot(p, w, 'r-', linewidth=3, label='Current γ')
        ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Linear (Rational)')
        
        if show_research:
            w_tk = probability_weighting_function(p, 0.61)
            ax2.plot(p, w_tk, 'g--', linewidth=2, alpha=0.7, label='Tversky & Kahneman (1992)')
            
        ax2.set_xlabel('Objective Probability')
        ax2.set_ylabel('Decision Weight')
        ax2.set_title('Probability Weighting Function')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        gains = np.linspace(0, 100, 50)
        losses = -gains
        gain_values = prospect_value_function(gains, alpha, beta, lambda_param)
        loss_values = prospect_value_function(losses, alpha, beta, lambda_param)
        
        ax3.plot(gains, gain_values, 'g-', linewidth=3, label='Gains')
        ax3.plot(-losses, loss_values, 'r-', linewidth=3, label='Losses')
        ax3.axhline(y=0, color='k', linestyle='-', alpha=0.3)
        ax3.set_xlabel('Magnitude')
        ax3.set_ylabel('Subjective Value')
        ax3.set_title('Loss Aversion Asymmetry')
        ax3.grid(True, alpha=0.3)
        ax3.legend()
        
        research_data = {
            'Study': ['Tversky & Kahneman (1992)', 'Camerer & Ho (1994)', 
                     'Wu & Gonzalez (1996)', 'Abdellaoui (2000)', 'Your Parameters'],
            'Alpha': [0.88, 0.56, 0.71, 0.89, alpha],
            'Beta': [0.88, 0.56, 0.84, 0.92, beta], 
            'Lambda': [2.25, 2.07, 1.38, 2.61, lambda_param],
            'Gamma': [0.61, 0.56, 0.60, 0.60, gamma]
        }
        
        df = pd.DataFrame(research_data)
        
        x_pos = np.arange(len(df))
        width = 0.2
        
        ax4.bar(x_pos - 1.5*width, df['Alpha'], width, label='α', alpha=0.8)
        ax4.bar(x_pos - 0.5*width, df['Beta'], width, label='β', alpha=0.8)
        ax4.bar(x_pos + 0.5*width, df['Lambda']/3, width, label='λ/3', alpha=0.8)  # Scale lambda
        ax4.bar(x_pos + 1.5*width, df['Gamma'], width, label='γ', alpha=0.8)
        
        ax4.set_xlabel('Studies')
        ax4.set_ylabel('Parameter Values')
        ax4.set_title('Parameter Estimates Across Studies')
        ax4.set_xticks(x_pos)
        ax4.set_xticklabels(df['Study'], rotation=45, ha='right')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"🧠 Current Parameter Analysis:")
        print(f"• Loss Aversion: A $10 loss feels like a ${10*lambda_param:.1f} gain")
        print(f"• Gain Sensitivity: α = {alpha:.2f} ({'more' if alpha < 0.88 else 'less'} curved than typical)")
        print(f"• Loss Sensitivity: β = {beta:.2f} ({'more' if beta < 0.88 else 'less'} curved than typical)")
        print(f"• Probability Distortion: γ = {gamma:.2f} ({'more' if gamma < 0.61 else 'less'} distorted than typical)")
        
        prob_10 = probability_weighting_function(0.1, gamma)
        prob_50 = probability_weighting_function(0.5, gamma)
        prob_90 = probability_weighting_function(0.9, gamma)
        
        print(f"\n📊 Probability Perception Examples:")
        print(f"• 10% feels like {prob_10*100:.1f}% (overweighted by {(prob_10-0.1)*100:.1f} points)")
        print(f"• 50% feels like {prob_50*100:.1f}% ({'under' if prob_50 < 0.5 else 'over'}weighted by {abs(prob_50-0.5)*100:.1f} points)")
        print(f"• 90% feels like {prob_90*100:.1f}% (underweighted by {(0.9-prob_90)*100:.1f} points)")

comprehensive_prospect_theory_demo()




*"Roughly speaking, the psychological principle that underlies the endowment effect is that losses loom larger than gains."* - Richard Thaler


Loss aversion is perhaps the most robust finding in behavioral economics. The basic principle: **the pain of losing is psychologically about twice as powerful as the pleasure of gaining**.


In prospect theory, loss aversion is captured by the parameter $\lambda$ in the value function:

$$v(x) = \begin{cases}
x^\alpha & \text{if } x \geq 0 \\
-\lambda(-x)^\beta & \text{if } x < 0
\end{cases}$$

Where $\lambda > 1$ indicates loss aversion. Typical estimates range from 1.5 to 4.0, with a median around 2.25.


**Thaler (1980)** first documented this: people value things more highly when they own them.

**Classic Experiment** (Kahneman, Knetsch & Thaler, 1990):
- Give half the participants coffee mugs
- Allow trading between mug owners and non-owners
- **Prediction**: ~50% should trade (random allocation)
- **Result**: Only ~10% traded!

**WTA/WTP Ratio**: Willingness to Accept / Willingness to Pay
$$\text{Ratio} = \frac{\text{WTA}}{\text{WTP}} = 1 + \frac{1}{\lambda}$$

**Samuelson & Zeckhauser (1988)**: People stick with current options even when better alternatives exist.

**401(k) Example**: Employees rarely change default investment options, even when clearly suboptimal.

**Shefrin & Statman (1985)**: Investors hold losing stocks too long and sell winning stocks too soon.

**Mechanism**: Selling a winner realizes a gain (good), selling a loser realizes a loss (painful).


- **Ordinary Goods**: WTA/WTP ratio ≈ 2.6
- **Non-market Goods**: WTA/WTP ratio ≈ 7.2  
- **Public Goods**: WTA/WTP ratio ≈ 10.4

**Maddux et al. (2010)**: Loss aversion varies across cultures
- **East Asians**: Lower loss aversion (more holistic thinking)
- **Americans**: Higher loss aversion (more analytic thinking)

**Tom et al. (2007)**: fMRI studies show:
- **Gains**: Activate reward regions (ventral striatum)
- **Losses**: Activate loss regions (amygdala, anterior insula) more strongly
- **Ratio**: Neural loss response ~2x stronger than gain response


**Madrian & Shea (2001)**: 401(k) auto-enrollment
- **Opt-in**: 37% participation
- **Opt-out**: 86% participation

**Allcott (2011)**: Home energy reports
- Showing neighbors' usage reduces consumption by 2-3%
- Loss framing ("you're using more") more effective than gain framing

**McCaffery & Baron (2006)**: 
- People prefer tax "rebates" over equivalent "bonus"
- Same money, different reference point


In [ ]:

def advanced_loss_aversion_analysis():
    """
    Comprehensive loss aversion analysis with multiple experimental paradigms
    """
    
    @widgets.interact(
        experiment_type=widgets.Dropdown(
            options=['Endowment Effect', 'Disposition Effect', 'Status Quo Bias', 'Reference Point Shifts'],
            value='Endowment Effect',
            description='Experiment:'
        ),
        lambda_param=widgets.FloatSlider(value=2.25, min=1.0, max=5.0, step=0.1, 
                                        description='Loss Aversion (λ)'),
        n_subjects=widgets.IntSlider(value=1000, min=100, max=2000, step=100,
                                    description='Sample Size'),
        noise_level=widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05,
                                       description='Individual Variation')
    )
    def run_experiment(experiment_type, lambda_param, n_subjects, noise_level):
        np.random.seed(42)
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        
        if experiment_type == 'Endowment Effect':
            true_value = 10  # True value of mug
            
            individual_lambdas = np.random.normal(lambda_param, noise_level, n_subjects)
            individual_lambdas = np.maximum(individual_lambdas, 1.0)  # Ensure λ ≥ 1
            
            owners = np.random.choice([True, False], size=n_subjects)
            
            wta_values = []
            wtp_values = []
            
            for i in range(n_subjects):
                individual_noise = np.random.normal(0, noise_level * true_value)
                perceived_value = true_value + individual_noise
                
                if owners[i]:
                    wta = perceived_value * (1 + 1/individual_lambdas[i])
                    wta_values.append(wta)
                else:
                    wtp = perceived_value
                    wtp_values.append(wtp)
            
            trades = 0
            for wta in wta_values:
                for wtp in wtp_values:
                    if wtp >= wta:
                        trades += 1
                        break
            
            trade_rate = trades / len(wta_values) * 100
            actual_ratio = np.mean(wta_values) / np.mean(wtp_values)
            
            ax1.hist(wta_values, alpha=0.7, label=f'WTA (Owners)', bins=30, color=colors['loss'], density=True)
            ax1.hist(wtp_values, alpha=0.7, label=f'WTP (Buyers)', bins=30, color=colors['gain'], density=True)
            ax1.axvline(np.mean(wta_values), color=colors['loss'], linestyle='--', linewidth=2)
            ax1.axvline(np.mean(wtp_values), color=colors['gain'], linestyle='--', linewidth=2)
            ax1.set_xlabel('Valuation ($)')
            ax1.set_ylabel('Density')
            ax1.set_title('Endowment Effect: WTA vs WTP Distributions')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
            
            lambda_range = np.linspace(1, 5, 50)
            theoretical_ratios = 1 + 1/lambda_range
            
            ax2.plot(lambda_range, theoretical_ratios, 'b-', linewidth=3, label='Theoretical')
            ax2.scatter([lambda_param], [actual_ratio], color='red', s=100, zorder=5, label=f'Observed: {actual_ratio:.2f}')
            ax2.set_xlabel('Loss Aversion Parameter (λ)')
            ax2.set_ylabel('WTA/WTP Ratio')
            ax2.set_title('Theoretical vs Observed Ratios')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
            
            research_ratios = {
                'Kahneman et al. (1990) - Mugs': 2.2,
                'Knetsch (1989) - Tokens': 2.6,
                'Franciosi et al. (1996) - Induced Value': 1.3,
                'List (2003) - Sports Cards': 1.8,
                'Current Simulation': actual_ratio
            }
            
            studies = list(research_ratios.keys())
            ratios = list(research_ratios.values())
            colors_bar = ['skyblue'] * (len(studies)-1) + ['red']
            
            ax3.bar(range(len(studies)), ratios, color=colors_bar, alpha=0.7)
            ax3.set_xlabel('Study')
            ax3.set_ylabel('WTA/WTP Ratio')
            ax3.set_title('Endowment Effect Across Studies')
            ax3.set_xticks(range(len(studies)))
            ax3.set_xticklabels(studies, rotation=45, ha='right')
            ax3.grid(True, alpha=0.3)
            
            lambda_values = np.linspace(1, 4, 20)
            trade_rates = []
            
            for lam in lambda_values:
                temp_wta = true_value * (1 + 1/lam) + np.random.normal(0, noise_level, 100)
                temp_wtp = true_value + np.random.normal(0, noise_level, 100)
                temp_trades = sum(1 for wta in temp_wta for wtp in temp_wtp if wtp >= wta) / len(temp_wta)
                trade_rates.append(temp_trades * 100)
            
            ax4.plot(lambda_values, trade_rates, 'g-', linewidth=3)
            ax4.axhline(50, color='red', linestyle='--', label='Rational Prediction (50%)')
            ax4.axvline(lambda_param, color='blue', linestyle=':', label=f'Current λ = {lambda_param:.2f}')
            ax4.set_xlabel('Loss Aversion Parameter (λ)')
            ax4.set_ylabel('Trading Rate (%)')
            ax4.set_title('Predicted Trading Rates')
            ax4.legend()
            ax4.grid(True, alpha=0.3)
            
            print(f"📊 Endowment Effect Results:")
            print(f"• {len(wta_values)} owners, {len(wtp_values)} buyers")
            print(f"• Mean WTA: ${np.mean(wta_values):.2f}")
            print(f"• Mean WTP: ${np.mean(wtp_values):.2f}")
            print(f"• WTA/WTP Ratio: {actual_ratio:.2f} (theory predicts {1 + 1/lambda_param:.2f})")
            print(f"• Trading Rate: {trade_rate:.1f}% (rational theory predicts 50%)")
            
        elif experiment_type == 'Disposition Effect':
            n_stocks = n_subjects
            purchase_prices = np.random.uniform(50, 150, n_stocks)
            current_prices = purchase_prices * np.random.lognormal(0, 0.3, n_stocks)
            
            gains_losses = current_prices - purchase_prices
            gains_losses_pct = gains_losses / purchase_prices * 100
            
            sell_probs = []
            for gl in gains_losses:
                if gl > 0:  # Gain
                    utility_sell = prospect_value_function(gl, 0.88, 0.88, lambda_param)
                    expected_future = 0.6 * prospect_value_function(gl * 1.1, 0.88, 0.88, lambda_param) + \
                                    0.4 * prospect_value_function(gl * 0.95, 0.88, 0.88, lambda_param)
                    prob_sell = expit(utility_sell - expected_future)  # Sigmoid function
                else:  # Loss
                    utility_sell = prospect_value_function(gl, 0.88, 0.88, lambda_param)
                    expected_future = 0.4 * prospect_value_function(gl * 1.1, 0.88, 0.88, lambda_param) + \
                                    0.6 * prospect_value_function(gl * 0.9, 0.88, 0.88, lambda_param)
                    prob_sell = expit(utility_sell - expected_future)
                
                sell_probs.append(prob_sell)
            
            sell_probs = np.array(sell_probs)
            
            winners = gains_losses > 0
            losers = gains_losses < 0
            
            ax1.scatter(gains_losses_pct[winners], sell_probs[winners], 
                       alpha=0.6, color=colors['gain'], label='Winners', s=30)
            ax1.scatter(gains_losses_pct[losers], sell_probs[losers], 
                       alpha=0.6, color=colors['loss'], label='Losers', s=30)
            ax1.set_xlabel('Gain/Loss (%)')
            ax1.set_ylabel('Probability of Selling')
            ax1.set_title('Disposition Effect: Selling Probability vs Performance')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
            
            if np.sum(winners) > 0 and np.sum(losers) > 0:
                avg_sell_winners = np.mean(sell_probs[winners])
                avg_sell_losers = np.mean(sell_probs[losers])
                
                ax2.bar(['Winners', 'Losers'], [avg_sell_winners, avg_sell_losers], 
                       color=[colors['gain'], colors['loss']], alpha=0.7)
                ax2.set_ylabel('Average Selling Probability')
                ax2.set_title('Disposition Effect Summary')
                ax2.grid(True, alpha=0.3)
                
                print(f"📊 Disposition Effect Results:")
                print(f"• {np.sum(winners)} winning stocks, {np.sum(losers)} losing stocks")
                print(f"• Average selling probability for winners: {avg_sell_winners:.3f}")
                print(f"• Average selling probability for losers: {avg_sell_losers:.3f}")
                print(f"• Disposition ratio: {avg_sell_winners/avg_sell_losers:.2f}")
                print(f"• Higher ratio = stronger disposition effect")
        
        plt.tight_layout()
        plt.show()

advanced_loss_aversion_analysis()
